# Construcción OBT (One Big Table)

Lee de RAW (TRIPS_YELLOW + TRIPS_GREEN), unifica esquemas, enriquece con lookups (taxi zones, catalogos), agrega derivadas y escribe a `ANALYTICS.OBT_TRIPS`.

- Grano: 1 fila = 1 viaje
- Idempotencia: DELETE + INSERT por source_year/source_month
- Procesamiento: mes a mes para no saturar memoria

## Imports y configuración

In [ ]:
import os
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import snowflake.connector
from tqdm.notebook import tqdm as tqdm_notebook

SNOWFLAKE_ACCOUNT   = os.environ["SNOWFLAKE_ACCOUNT"]
SNOWFLAKE_USER      = os.environ["SNOWFLAKE_USER"]
SNOWFLAKE_PASSWORD  = os.environ["SNOWFLAKE_PASSWORD"]
SNOWFLAKE_DATABASE  = os.environ["SNOWFLAKE_DATABASE"]
SNOWFLAKE_WAREHOUSE = os.environ["SNOWFLAKE_WAREHOUSE"]
SNOWFLAKE_ROLE      = os.environ.get("SNOWFLAKE_ROLE", "SYSADMIN")
SCHEMA_RAW          = os.environ.get("SNOWFLAKE_SCHEMA_RAW", "RAW")
SCHEMA_ANALYTICS    = os.environ.get("SNOWFLAKE_SCHEMA_ANALYTICS", "ANALYTICS")

START_YEAR  = int(os.environ.get("START_YEAR", "2015"))
END_YEAR    = int(os.environ.get("END_YEAR", "2025"))
RUN_ID      = os.environ.get("RUN_ID", "run_001")

VALIDATE_NULLS      = os.environ.get("VALIDATE_NULLS", "true").lower() == "true"
VALIDATE_RANGES     = os.environ.get("VALIDATE_RANGES", "true").lower() == "true"
VALIDATE_TIMESTAMPS = os.environ.get("VALIDATE_TIMESTAMPS", "true").lower() == "true"

spark = SparkSession.builder \
    .appName("P3_Construccion_OBT") \
    .config("spark.jars.packages",
            "net.snowflake:spark-snowflake_2.12:2.16.0-spark_3.4,"
            "net.snowflake:snowflake-jdbc:3.16.1") \
    .getOrCreate()

sf_raw = {
    "sfURL": f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com",
    "sfUser": SNOWFLAKE_USER, "sfPassword": SNOWFLAKE_PASSWORD,
    "sfDatabase": SNOWFLAKE_DATABASE, "sfSchema": SCHEMA_RAW,
    "sfWarehouse": SNOWFLAKE_WAREHOUSE, "sfRole": SNOWFLAKE_ROLE,
}

sf_analytics = {
    "sfURL": f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com",
    "sfUser": SNOWFLAKE_USER, "sfPassword": SNOWFLAKE_PASSWORD,
    "sfDatabase": SNOWFLAKE_DATABASE, "sfSchema": SCHEMA_ANALYTICS,
    "sfWarehouse": SNOWFLAKE_WAREHOUSE, "sfRole": SNOWFLAKE_ROLE,
}

def get_snowflake_connection(schema=SCHEMA_ANALYTICS):
    return snowflake.connector.connect(
        account=SNOWFLAKE_ACCOUNT, user=SNOWFLAKE_USER,
        password=SNOWFLAKE_PASSWORD, database=SNOWFLAKE_DATABASE,
        schema=schema, warehouse=SNOWFLAKE_WAREHOUSE, role=SNOWFLAKE_ROLE,
    )

print(f"Spark {spark.version} | OBT: {SNOWFLAKE_DATABASE}.{SCHEMA_ANALYTICS}.OBT_TRIPS")
print(f"Rango: {START_YEAR}-{END_YEAR} | RUN_ID: {RUN_ID}")

## Cargar lookups desde RAW

In [ ]:
df_zones = spark.read.format("net.snowflake.spark.snowflake") \
    .options(**sf_raw).option("dbtable", "TAXI_ZONES").load()
df_zones = df_zones.toDF(*[c.lower() for c in df_zones.columns])

df_payment = spark.read.format("net.snowflake.spark.snowflake") \
    .options(**sf_raw).option("dbtable", "PAYMENT_TYPES").load()
df_payment = df_payment.toDF(*[c.lower() for c in df_payment.columns])

df_rate = spark.read.format("net.snowflake.spark.snowflake") \
    .options(**sf_raw).option("dbtable", "RATE_CODES").load()
df_rate = df_rate.toDF(*[c.lower() for c in df_rate.columns])

df_vendor = spark.read.format("net.snowflake.spark.snowflake") \
    .options(**sf_raw).option("dbtable", "VENDORS").load()
df_vendor = df_vendor.toDF(*[c.lower() for c in df_vendor.columns])

# Broadcast (tablas pequenas, optimiza joins)
df_zones_b = F.broadcast(df_zones)
df_payment_b = F.broadcast(df_payment)
df_rate_b = F.broadcast(df_rate)
df_vendor_b = F.broadcast(df_vendor)

print(f"Lookups cargados: zones={df_zones.count()}, payment={df_payment.count()}, rate={df_rate.count()}, vendor={df_vendor.count()}")

## Funciones de construcción OBT

In [ ]:
# Columnas comunes que ambos servicios deben tener
UNIFIED_COLS = [
    "pickup_datetime", "dropoff_datetime", "vendorid", "passenger_count",
    "trip_distance", "ratecodeid", "store_and_fwd_flag", "pulocationid",
    "dolocationid", "payment_type", "fare_amount", "extra", "mta_tax",
    "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount",
    "congestion_surcharge", "airport_fee", "trip_type",
    "service_type", "run_id", "source_year", "source_month",
    "source_path", "ingested_at_utc"
]


def read_raw_chunk(service, year, month):
    """Lee un mes de un servicio desde RAW. Retorna None si no hay datos."""
    table = f"TRIPS_{service.upper()}"
    try:
        df = spark.read.format("net.snowflake.spark.snowflake") \
            .options(**sf_raw) \
            .option("query", f"""
                SELECT * FROM {table}
                WHERE source_year = {year} AND source_month = {month}
            """).load()
        df = df.toDF(*[c.lower() for c in df.columns])
        if df.head(1):
            return df
    except Exception:
        pass
    return None


def unify_schema(df):
    """Agrega columnas faltantes como NULL para que yellow y green tengan el mismo esquema."""
    for col in UNIFIED_COLS:
        if col not in df.columns:
            df = df.withColumn(col, F.lit(None))
    return df.select(UNIFIED_COLS)


def enrich_and_derive(df):
    """Joins con lookups + columnas derivadas para la OBT."""
    # Join Taxi Zones (pickup y dropoff)
    pu_zones = df_zones_b.select(
        F.col("locationid").alias("pu_loc_join"),
        F.col("zone").alias("pu_zone"),
        F.col("borough").alias("pu_borough")
    )
    do_zones = df_zones_b.select(
        F.col("locationid").alias("do_loc_join"),
        F.col("zone").alias("do_zone"),
        F.col("borough").alias("do_borough")
    )

    df = df.join(pu_zones, df["pulocationid"] == pu_zones["pu_loc_join"], "left").drop("pu_loc_join")
    df = df.join(do_zones, df["dolocationid"] == do_zones["do_loc_join"], "left").drop("do_loc_join")

    # Join catalogos
    df = df.join(df_payment_b, "payment_type", "left")
    df = df.join(
        df_rate_b,
        df["ratecodeid"].cast("int") == df_rate_b["rate_code_id"], "left"
    ).drop("rate_code_id")
    df = df.join(
        df_vendor_b,
        df["vendorid"].cast("int") == df_vendor_b["vendor_id"], "left"
    ).drop("vendor_id")

    # Columnas de tiempo derivadas
    df = df.withColumn("pickup_date", F.to_date("pickup_datetime")) \
           .withColumn("pickup_hour", F.hour("pickup_datetime")) \
           .withColumn("dropoff_date", F.to_date("dropoff_datetime")) \
           .withColumn("dropoff_hour", F.hour("dropoff_datetime")) \
           .withColumn("day_of_week", F.dayofweek("pickup_datetime")) \
           .withColumn("month", F.month("pickup_datetime")) \
           .withColumn("year", F.year("pickup_datetime"))

    # Derivadas de viaje
    df = df.withColumn("trip_duration_min",
        (F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / 60.0
    )
    df = df.withColumn("avg_speed_mph",
        F.when(
            (F.col("trip_duration_min") > 0) & (F.col("trip_distance") > 0),
            F.col("trip_distance") / (F.col("trip_duration_min") / 60.0)
        )
    )
    df = df.withColumn("tip_pct",
        F.when(
            F.col("total_amount") > 0,
            (F.col("tip_amount") / F.col("total_amount")) * 100.0
        )
    )

    # Filtrar segun flags de validacion (aqui si se filtran, no en RAW)
    if VALIDATE_NULLS:
        df = df.filter(F.col("pickup_datetime").isNotNull() & F.col("dropoff_datetime").isNotNull())
    if VALIDATE_TIMESTAMPS:
        df = df.filter(F.col("pickup_datetime") <= F.col("dropoff_datetime"))
    if VALIDATE_RANGES:
        df = df.filter((F.col("trip_distance") >= 0) | F.col("trip_distance").isNull())
        df = df.filter((F.col("total_amount") >= 0) | F.col("total_amount").isNull())

    # Renombrar para esquema OBT final
    df = df.withColumnRenamed("pulocationid", "pu_location_id") \
           .withColumnRenamed("dolocationid", "do_location_id") \
           .withColumnRenamed("vendorid", "vendor_id") \
           .withColumnRenamed("ratecodeid", "rate_code_id")

    return df


def delete_obt_partition(year, month):
    """Idempotencia: borra datos del mes en OBT antes de reinsertar."""
    conn = get_snowflake_connection()
    try:
        cursor = conn.cursor()
        cursor.execute(f"""
            DELETE FROM {SCHEMA_ANALYTICS}.OBT_TRIPS
            WHERE source_year = {year} AND source_month = {month}
        """)
        return cursor.rowcount
    except Exception:
        return 0
    finally:
        conn.close()


print("Funciones OBT cargadas.")

## Construcción OBT mes a mes

Para cada mes: lee yellow+green de RAW, unifica, enriquece, agrega derivadas y escribe a ANALYTICS.OBT_TRIPS.

In [ ]:
obt_log = []
chunks = [(y, m) for y in range(START_YEAR, END_YEAR + 1) for m in range(1, 13)]

pbar = tqdm_notebook(chunks, desc="Construccion OBT", unit="mes")

for year, month in pbar:
    pbar.set_postfix_str(f"{year}-{month:02d}")
    start = time.time()

    # Leer ambos servicios
    dfs = []
    for svc in ["yellow", "green"]:
        chunk = read_raw_chunk(svc, year, month)
        if chunk is not None:
            dfs.append(unify_schema(chunk))

    if not dfs:
        obt_log.append({"year": year, "month": month, "status": "missing", "rows": 0})
        continue

    try:
        # Unir yellow + green
        df_unified = dfs[0]
        for extra in dfs[1:]:
            df_unified = df_unified.unionByName(extra, allowMissingColumns=True)

        # Enriquecer + derivadas + filtros
        df_obt = enrich_and_derive(df_unified)
        row_count = df_obt.count()

        # Idempotencia
        deleted = delete_obt_partition(year, month)
        if deleted > 0:
            print(f"  IDEMPOTENCIA [{year}-{month:02d}]: borradas {deleted:,} filas previas")

        # Escribir a OBT
        df_obt.write.format("net.snowflake.spark.snowflake") \
            .options(**sf_analytics) \
            .option("dbtable", "OBT_TRIPS") \
            .mode("append") \
            .save()

        elapsed = round(time.time() - start, 2)
        obt_log.append({"year": year, "month": month, "status": "ok", "rows": row_count, "time_sec": elapsed})
        pbar.set_postfix_str(f"{year}-{month:02d} | {row_count:,} filas | {elapsed}s")

    except Exception as e:
        obt_log.append({"year": year, "month": month, "status": "failed", "rows": 0, "error": str(e)})
        print(f"  ERROR [{year}-{month:02d}]: {e}")

pbar.close()
ok = sum(1 for r in obt_log if r["status"] == "ok")
total = sum(r["rows"] for r in obt_log)
print(f"\nOBT finalizada: {ok} meses ok | {total:,} filas totales")

## Verificación: muestra de OBT_TRIPS

In [ ]:
df_sample = spark.read.format("net.snowflake.spark.snowflake") \
    .options(**sf_analytics) \
    .option("query", "SELECT * FROM OBT_TRIPS LIMIT 5") \
    .load()

print(f"Columnas OBT: {len(df_sample.columns)}")
for c in sorted(df_sample.columns):
    print(f"  {c}")

df_sample.show(5, truncate=False, vertical=True)

## Conteos por servicio y anio

In [ ]:
df_counts = spark.read.format("net.snowflake.spark.snowflake") \
    .options(**sf_analytics) \
    .option("query", """
        SELECT service_type, source_year, COUNT(*) as rows
        FROM OBT_TRIPS
        GROUP BY service_type, source_year
        ORDER BY service_type, source_year
    """).load()

df_counts.show(50, truncate=False)